In [6]:
%load[qpepennylane.py]

UsageError: Line magic function `%load[qpepennylane.py]` not found.


In [3]:
# %load qpe.py
import numpy as np

def hadamard(n):
   """Creates an n-qubit Hadamard gate as a matrix."""
   H = np.array([[1, 1], [1, -1]]) / np.sqrt(2)
   H_n = H
   for _ in range(n - 1):
       H_n = np.kron(H_n, H)  # Tensor product to expand Hadamard gate
   return H_n

def qft(n):
   """Creates an n-qubit Quantum Fourier Transform (QFT) matrix."""
   N = 2**n
   omega = np.exp(2j * np.pi / N)
   qft_matrix = np.array([[omega**(i * j) for j in range(N)] for i in range(N)]) / np.sqrt(N)
   return qft_matrix

def inverse_qft(n):
   """Creates an n-qubit inverse Quantum Fourier Transform (QFT†) matrix."""
   return np.conj(qft(n)).T  # Hermitian transpose of QFT

def controlled_unitary(U, control, target, n):
   """Creates an n-qubit controlled unitary matrix."""
   I = np.eye(2**n)  # Identity matrix
   CU = np.copy(I)
   for i in range(2**n):
       if (i >> control) & 1:  # Check if control qubit is |1⟩
           CU[i, :] = np.kron(np.eye(2**target), U).dot(I[i, :])
   return CU

def apply_gate(state, gate):
   """Applies a gate (matrix) to a quantum state (vector)."""
   return gate @ state

def measure(state):
   """Simulates measurement by computing probability distribution."""
   probabilities = np.abs(state) ** 2
   return np.argmax(probabilities)  # Return the most probable outcome

def qpe(unitary, phi, num_counting_qubits):
   """Simulates the Quantum Phase Estimation algorithm."""
   n = num_counting_qubits
   total_qubits = n + 1
   dim = 2**total_qubits

   # Step 1: Initialize state |0...0⟩ ⊗ |ψ⟩
   state = np.zeros(dim, dtype=complex)
   state[0] = 1  # |00...0⟩

   # Step 2: Apply Hadamard to counting qubits
   H_n = hadamard(n)
   state = apply_gate(state.reshape(2**n, 2), H_n).reshape(dim)

   # Step 3: Apply controlled-U^2^j operations
   for j in range(n):
       power = 2**j
       U_power = np.linalg.matrix_power(unitary, power)
       CU = controlled_unitary(U_power, j, n, total_qubits)
       state = apply_gate(state, CU)

   # Step 4: Apply inverse QFT
   IQFT = inverse_qft(n)
   state = apply_gate(state.reshape(2**n, 2), IQFT).reshape(dim)

   # Step 5: Measure and return estimated phase
   measurement_result = measure(state)
   return measurement_result / (2**n)  # Convert binary to decimal

# Define the unitary U with phase φ = 1/3
phi = 1/3
U = np.array([[1, 0], [0, np.exp(2j * np.pi * phi)]])  # Phase gate

# Run QPE
num_counting_qubits = 3  # More qubits give higher precision
estimated_phi = qpe(U, phi, num_counting_qubits)

print(f"Estimated phase: {estimated_phi}")
print(f"Actual phase: {phi}")


Estimated phase: 0.0
Actual phase: 0.3333333333333333


/var/folders/td/3yk470mj5p931p9dtkk0y6jw0000gn/T/ipykernel_52733/382557020.py:29: ComplexWarning: Casting complex values to real discards the imaginary part
  CU[i, :] = np.kron(np.eye(2**target), U).dot(I[i, :])


In [8]:
# %load qpepennylane.py
import pennylane as qml
import numpy as np

def qft(n):
   """Applies the inverse Quantum Fourier Transform (QFT†) circuit."""
   for i in range(n):
       qml.Hadamard(wires=i)
       for j in range(i):
           qml.CPhase(-np.pi / (2 ** (i - j)), wires=[j, i])
   for i in range(n // 2):
       qml.SWAP(wires=[i, n - i - 1])

def controlled_unitary(U, control, target):
   """Applies a controlled-unitary operation."""
   qml.ctrl(U, control=control)(wires=target)


def qpe(phi, num_counting_qubits):
   """Quantum Phase Estimation circuit in PennyLane."""
   total_qubits = num_counting_qubits + 1

   dev = qml.device("default.qubit", wires=total_qubits, shots=1000)

   @qml.qnode(dev)
   def circuit():
       # Initialize the counting register in |0> and eigenstate register in |1>
       qml.PauliX(wires=num_counting_qubits)  # Set last qubit to |1>

       # Apply Hadamard to counting qubits
       for qubit in range(num_counting_qubits):
           qml.Hadamard(wires=qubit)

       # Apply controlled-U^2^j operations
       for j in range(num_counting_qubits):
           power = 2**j
           U = qml.RZ(2 * np.pi * phi, wires=num_counting_qubits)  # Phase shift
           controlled_unitary(U, control=j, target=num_counting_qubits)

       # Apply inverse QFT
       qft(num_counting_qubits)

       # Measure counting qubits
       return qml.sample(wires=range(num_counting_qubits))

   # Run the circuit
   samples = circuit()

   # Convert measurement results to decimal phase estimate
   binary_result = "".join(map(str, samples[0]))  # Take the first sample
   estimated_phi = int(binary_result, 2) / (2 ** num_counting_qubits)

   return estimated_phi

# Define the phase φ
phi = 1/3

# Number of counting qubits (higher gives better precision)
num_counting_qubits = 3

# Run QPE
estimated_phi = qpe(phi, num_counting_qubits)

print(f"Estimated phase: {estimated_phi}")
print(f"Actual phase: {phi}")


TypeError: 'CRZ' object is not callable

In [12]:
# %load qpeqiskit.py
from qiskit import QuantumCircuit, Aer, transpile, assemble, execute
from qiskit.visualization import plot_histogram
import numpy as np

def qpe(unitary, num_counting_qubits):
   """Quantum Phase Estimation Algorithm.

   Args:
       unitary (QuantumCircuit): The unitary operator whose phase we estimate.
       num_counting_qubits (int): Number of qubits in the counting register.

   Returns:
       QuantumCircuit: QPE quantum circuit
   """
   n = num_counting_qubits
   qc = QuantumCircuit(n + 1, n)  # n counting qubits + 1 eigenstate qubit

   # Step 1: Apply Hadamard to counting qubits
   for qubit in range(n):
       qc.h(qubit)

   # Step 2: Apply controlled-U^2^j operations
   for j in range(n):
       power = 2**j
       controlled_U = unitary.control(1).power(power)
       qc.append(controlled_U, [j] + [n])  # Control: j, Target: n

   # Step 3: Apply inverse QFT
   qc.append(qft_dagger(n), range(n))

   # Step 4: Measure counting qubits
   qc.measure(range(n), range(n))

   return qc

def qft_dagger(n):
   """Creates an inverse Quantum Fourier Transform (QFT†) circuit."""
   qc = QuantumCircuit(n)
   for qubit in range(n//2):
       qc.swap(qubit, n-qubit-1)

   for j in range(n):
       for m in range(j):
           qc.cp(-np.pi / (2**(j-m)), m, j)
       qc.h(j)

   return qc

# Define the unitary U with phase phi = 1/3
phi = 1/3
U = QuantumCircuit(1)
U.p(2 * np.pi * phi, 0)  # Phase gate

# Number of counting qubits
num_counting_qubits = 3

# Generate QPE circuit
qpe_circuit = qpe(U, num_counting_qubits)

# Simulate the circuit
simulator = Aer.get_backend('qasm_simulator')
compiled_circuit = transpile(qpe_circuit, simulator)
qobj = assemble(compiled_circuit)
result = simulator.run(qobj).result()

# Get measurement results
counts = result.get_counts()

# Plot histogram of results
plot_histogram(counts)


ImportError: cannot import name 'QuantumCircuit' from 'qiskit' (unknown location)